In [ ]:
import os
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from minusface import MinusBackbone
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import insightface
import wandb

wandb.init(project="student_distill")

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# load buffalo_l ONNX teacher
# ----------------------------
teacher_app = insightface.app.FaceAnalysis(name="buffalo_l")
teacher_app.prepare(ctx_id=0)
teacher_onnx = teacher_app.models["recognition"]

# ----------------------------
# student model with extra fc
# ----------------------------
class StudentHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = InceptionResnetV1(pretrained='vggface2')
        self.fc = nn.Linear(512, 512)

    def forward(self, x):
        e = self.base(x)
        return self.fc(e)

student = StudentHead().to(device).train()

for m in student.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        for p in m.parameters():
            p.requires_grad = False

# ----------------------------
# conversion model
# ----------------------------
conversion_model = MinusBackbone(mode='stage1')
conversion_model.load_state_dict(torch.load("../../../../minusface_stage1.pth", map_location='cpu'))
conversion_model = conversion_model.eval().to(device)

# ----------------------------
# transforms
# ----------------------------
tf_teacher_raw = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor()
])

tf_student_ready = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

tf_conv = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor()
])

# ----------------------------
# dataset
# ----------------------------
def list_images(root):
    out = []
    for r,_,fs in os.walk(root):
        for f in fs:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                out.append(os.path.join(r,f))
    return out

class FaceDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        t_raw = tf_teacher_raw(img)
        c_raw = tf_conv(img)
        return t_raw, c_raw

# ----------------------------
# conversion model per batch
# ----------------------------
def convert_batch(conv_raw):
    conv_raw = conv_raw.to(device)
    with torch.no_grad():
        out = conversion_model(conv_raw)
    out = out[5]
    out = out.permute(0,2,3,1).cpu().numpy()
    out = (out - out.min((1,2,3),keepdims=True)) / (out.max((1,2,3),keepdims=True) - out.min((1,2,3),keepdims=True) + 1e-8)
    out = out[...,:3]
    arr = (out*255).astype(np.uint8)
    imgs = [Image.fromarray(x) for x in arr]
    return torch.stack([tf_student_ready(im) for im in imgs]).to(device)

# ----------------------------
# teacher per-image embedding
# ----------------------------
def teacher_emb_single(t_raw):
    arr = t_raw.numpy() * 255.0
    arr = arr.astype(np.uint8)
    arr = np.transpose(arr, (1,2,0))
    emb = teacher_onnx.get_feat(arr)
    return torch.tensor(emb, dtype=torch.float32)

def teacher_emb_batch(t_raw):
    out = []
    for i in range(t_raw.size(0)):
        out.append(teacher_emb_single(t_raw[i]))
    return torch.stack(out).to(device)

# ----------------------------
# loss
# ----------------------------
def loss_fn(s, t):
    s = F.normalize(s, dim=1)
    t = F.normalize(t, dim=1)
    return 0.5*F.mse_loss(s,t) + 0.5*(1 - (s*t).sum(dim=1)).mean()

# ----------------------------
# training
# ----------------------------
paths = list_images("training")
dataset = FaceDataset(paths)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=32, pin_memory=True)

epochs = 4
optimizer = torch.optim.AdamW(student.parameters(), lr=2e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=len(loader)*epochs)

for e in range(epochs):
    total = 0
    count = 0
    for t_raw, conv_raw in tqdm(loader):
        t_emb = teacher_emb_batch(t_raw)
        s_img = convert_batch(conv_raw)
        s_emb = student(s_img)
        loss = loss_fn(s_emb, t_emb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total += loss.item()
        count += 1
        wandb.log({"loss": loss.item(), "lr": scheduler.get_last_lr()[0]})
    torch.save(student.state_dict(), "student.pth")
    wandb.log({"epoch_loss": total/count})


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/user/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/user/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/user/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/user/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/user/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size

 43%|████▎     | 269/624 [27:42<36:48,  6.22s/it]  